# 0824_lsw_003_structure_comparison

**통합모델 vs 검사유형별 5분리 모델** 구조 비교 노트북입니다. 데이터 전처리(중복 제거, 시간순 6:2:2 분할)는 `main`의 `0824_kimjaehak_005_xgboost_baseline`과 동일하게 맞춰서, 구조 하나만 바뀌었을 때의 차이만 순수하게 비교합니다.

이번 단계에서는 `mapping.json` 기반 피처 마스킹을 적용하지 않습니다(구조와 별개 변수라 섞으면 원인을 구분할 수 없음) — 대신 학습 데이터 기준 상수열만 제거합니다. `mapping.json` 마스킹은 다음 단계(feature selection)에서 다룹니다.

## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0824_lsw_003_structure_comparison"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
MODEL_DIR = Path("../models")

# docs/lsw/project/scope_and_roadmap.md에 정한 임시 총비용 시나리오
COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)

experiment: 0824_lsw_003_structure_comparison


## 2. 원본 데이터 로딩

`0824_kimjaehak_005_xgboost_baseline`과 동일: 첫 번째 열을 `record_id`로 명시합니다.

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]

if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."
print("shape:", raw_df.shape)

shape: (440274, 78)


## 3. 완전 중복 행 제거

`record_id`와 `timestamp`를 제외한 나머지 전체 컬럼 기준으로 중복이면 첫 행만 유지합니다(kimjaehak의 baseline과 동일 기준).

In [3]:
dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
duplicate_rows_removed = int(duplicate_mask.sum())

clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

assert not clean_df.duplicated(subset=dedup_columns, keep=False).any()

pd.Series(
    {
        "rows_before": len(raw_df),
        "duplicate_rows_removed": duplicate_rows_removed,
        "rows_after": len(clean_df),
    }
)

rows_before               440274
duplicate_rows_removed     48282
rows_after                391992
dtype: int64

## 4. 피처/타깃 준비

In [4]:
clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]
print("전체 피처 수:", len(feature_columns_all))
print("inspection_type 포함 여부:", "inspection_type" in feature_columns_all)

전체 피처 수: 75
inspection_type 포함 여부: True


## 5. 시간순 Train/Validation/Test 분할

kimjaehak의 baseline과 동일하게, 누적 행 수 기준 60/80% 지점의 timestamp를 경계로 잡되 같은 timestamp 그룹이 두 세트에 걸치지 않게 합니다. **이 경계는 전체 데이터(모든 검사유형 합산) 기준으로 한 번만 계산**하고, 두 구조(통합/5분리) 모두 이 경계를 그대로 씁니다 — 그래야 두 모델이 정확히 같은 시험 구간에서 평가됩니다.

In [5]:
timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()

train_end_position = int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))
valid_end_position = int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))
train_end_time = timestamp_group_sizes.index[train_end_position]
valid_end_time = timestamp_group_sizes.index[valid_end_position]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time

assert timestamps.loc[train_mask].max() < timestamps.loc[valid_mask].min()
assert timestamps.loc[valid_mask].max() < timestamps.loc[test_mask].min()

pd.DataFrame(
    [
        {"split": name, "rows": int(mask.sum()), "row_ratio_pct": mask.mean() * 100}
        for name, mask in [("train", train_mask), ("validation", valid_mask), ("test", test_mask)]
    ]
).set_index("split")

,rows,row_ratio_pct
split,,
train,235222,60.006837
validation,78374,19.993775
test,78396,19.999388


## 6. 평가 함수 (Slip Rate / Volume Reduction / 총비용 / 임계값 선택)

`0823_lsw_002_baseline`과 같은 정의를 재사용하되, 임계값 탐색은 `0.01` 간격 grid 대신 **실제 관측된 예측확률값을 후보로 쓰는 exact search**로 바꿨습니다 — grid 해상도가 너무 성겨서 놓치는 임계값이 있었던 문제(이전 baseline에서 확인)를 고치기 위함입니다.

In [6]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result

## 7. Arm A — 통합모델

`inspection_type`을 포함한 전체 피처를 그대로 입력으로 사용하는 단일 XGBoost입니다 (kimjaehak baseline과 같은 구성).

In [7]:
def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model():
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )


unified_train_df = clean_df.loc[train_mask]
unified_valid_df = clean_df.loc[valid_mask]
unified_test_df = clean_df.loc[test_mask]

unified_feature_columns = get_non_constant_columns(feature_columns_all, unified_train_df)
print(f"통합모델 피처 수: {len(unified_feature_columns)} / {len(feature_columns_all)}")

unified_model = build_model()
unified_model.fit(unified_train_df[unified_feature_columns], unified_train_df[TARGET])

unified_valid_proba = unified_model.predict_proba(unified_valid_df[unified_feature_columns])[:, 1]
unified_test_proba = unified_model.predict_proba(unified_test_df[unified_feature_columns])[:, 1]

unified_threshold = select_threshold(unified_valid_df[TARGET], unified_valid_proba)
unified_result = evaluate_at_threshold(unified_test_df[TARGET], unified_test_proba, unified_threshold)
unified_result["n_train"] = len(unified_train_df)
unified_result["n_features"] = len(unified_feature_columns)
pd.Series(unified_result, name="unified")

통합모델 피처 수: 69 / 75


threshold           1.785043e-12
tn                  6.500000e+01
fp                  7.608200e+04
fn                  5.000000e+00
tp                  2.244000e+03
slip_rate           2.223210e-03
volume_reduction    8.536121e-04
pr_auc              2.368043e-01
roc_auc             8.463156e-01
total_cost_1:10     7.613200e+04
total_cost_1:100    7.658200e+04
n_train             2.352220e+05
n_features          6.900000e+01
Name: unified, dtype: float64

## 8. Arm B — 검사유형별 5분리 모델

같은 Train/Val/Test 경계 안에서 `inspection_type`별로 subset을 나눠 각각 XGBoost를 학습합니다. `inspection_type` 자체는 각 모델 안에서 상수이므로 상수열 제거 단계에서 자동으로 빠집니다.

In [8]:
split_results = {}
split_models = {}
split_feature_columns = {}
split_test_true = []
split_test_pred = []

for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type

    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]

    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)

    model = build_model()
    model.fit(type_train_df[type_feature_columns], type_train_df[TARGET])

    valid_proba = model.predict_proba(type_valid_df[type_feature_columns])[:, 1]
    test_proba = model.predict_proba(type_test_df[type_feature_columns])[:, 1]

    threshold = select_threshold(type_valid_df[TARGET], valid_proba)
    result = evaluate_at_threshold(type_test_df[TARGET], test_proba, threshold)
    result["n_train"] = len(type_train_df)
    result["n_val_pos"] = int((type_valid_df[TARGET] == 1).sum())
    result["n_features"] = len(type_feature_columns)

    split_results[inspection_type] = result
    split_models[inspection_type] = model
    split_feature_columns[inspection_type] = type_feature_columns

    split_test_true.append(type_test_df[TARGET].to_numpy())
    split_test_pred.append((test_proba >= threshold).astype(int))

split_results_df = pd.DataFrame(split_results).T
split_results_df.index.name = "inspection_type"
split_results_df

,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,roc_auc,total_cost_1:10,total_cost_1:100,n_train,n_val_pos,n_features
inspection_type,,,,,,,,,,,,,,
0,0.000006,8857.0,7794.0,22.0,111.0,0.165414,0.531920,0.067265,0.746373,8014.0,9994.0,41961.0,8.0,47.0
1,0.000025,1869.0,8459.0,8.0,776.0,0.010204,0.180964,0.403144,0.875167,8539.0,9259.0,34041.0,239.0,34.0
2,0.000006,3492.0,14958.0,12.0,691.0,0.017070,0.189268,0.356827,0.887697,15078.0,16158.0,74910.0,32.0,24.0
3,0.000007,9700.0,20288.0,43.0,560.0,0.071310,0.323463,0.269568,0.839741,20718.0,24588.0,80792.0,24.0,22.0
4,0.000001,0.0,730.0,0.0,26.0,0.000000,0.000000,0.023284,0.259405,730.0,730.0,3518.0,60.0,24.0


## 9. 구조 비교 — 통합 vs 5분리 (Pooled)

5분리 모델의 test 예측을 전부 이어붙여, 통합모델과 같은 전체 test 모집단 기준으로 직접 비교합니다.

In [9]:
pooled_true = np.concatenate(split_test_true)
pooled_pred = np.concatenate(split_test_pred)

pooled_tn, pooled_fp, pooled_fn, pooled_tp = confusion_matrix(pooled_true, pooled_pred, labels=[0, 1]).ravel()
split_pooled_result = {
    "tn": int(pooled_tn),
    "fp": int(pooled_fp),
    "fn": int(pooled_fn),
    "tp": int(pooled_tp),
    "slip_rate": slip_rate(pooled_true, pooled_pred),
    "volume_reduction": volume_reduction(pooled_true, pooled_pred),
}
for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
    split_pooled_result[f"total_cost_{name}"] = total_cost(pooled_true, pooled_pred, cost_fp, cost_fn)

comparison_table = pd.DataFrame(
    [
        {
            "구조": "통합모델",
            "Slip Rate": unified_result["slip_rate"],
            "Volume Reduction": unified_result["volume_reduction"],
            "총비용(1:10)": unified_result["total_cost_1:10"],
            "총비용(1:100)": unified_result["total_cost_1:100"],
            "PR-AUC": unified_result["pr_auc"],
        },
        {
            "구조": "5분리모델(pooled)",
            "Slip Rate": split_pooled_result["slip_rate"],
            "Volume Reduction": split_pooled_result["volume_reduction"],
            "총비용(1:10)": split_pooled_result["total_cost_1:10"],
            "총비용(1:100)": split_pooled_result["total_cost_1:100"],
            "PR-AUC": np.nan,
        },
    ]
).set_index("구조")
comparison_table

,Slip Rate,Volume Reduction,총비용(1:10),총비용(1:100),PR-AUC
구조,,,,,
통합모델,0.002223,0.000854,76132,76582,0.236804
5분리모델(pooled),0.037795,0.314103,53079,60729,NaN


## 10. 모델 저장

통합모델 1개, 검사유형별 모델 5개를 각각 저장합니다. 모델 파일은 Git에서 제외됩니다.

In [10]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

unified_path = MODEL_DIR / f"{EXPERIMENT_ID}_unified.pkl"
joblib.dump(
    {"model": unified_model, "feature_columns": unified_feature_columns, "threshold": unified_threshold},
    unified_path,
)
print(f"saved: {unified_path}")

for inspection_type, model in split_models.items():
    model_path = MODEL_DIR / f"{EXPERIMENT_ID}_type{inspection_type}.pkl"
    joblib.dump(
        {
            "model": model,
            "feature_columns": split_feature_columns[inspection_type],
            "threshold": split_results[inspection_type]["threshold"],
        },
        model_path,
    )
    print(f"saved: {model_path}")

saved: ..\models\0824_lsw_003_structure_comparison_unified.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type0.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type1.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type2.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type3.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type4.pkl


## 11. 결론 및 다음 단계

### 결과 요약

| 구조 | Slip Rate | Volume Reduction | 총비용(1:10) | 총비용(1:100) |
|---|---:|---:|---:|---:|
| 통합모델 | 0.22% | 0.09% | 76,132 | 76,582 |
| 5분리모델(pooled) | 3.78% | 31.4% | 53,079 | 60,729 |

### 핵심 관찰

- **통합모델**: 안전 기준(Slip Rate ≤1%)은 만족하지만, 임계값이 사실상 0(1.79e-12)까지 밀려 Test negative 76,147건 중 65건만 자동 처리되고 나머지는 전부 수동검사로 감 — Volume Reduction 0.09%로 사실상 자동화 효과가 없다. `0824_kimjaehak_005_xgboost_baseline`과 동일한 데이터/모델을 재사용했으므로 test negative 총량(76,147)이 그 노트북과 정확히 일치함을 확인했다(재현성 검증).
- **5분리모델**: Volume Reduction이 31.4%로 크게 개선됐지만, **pooled Slip Rate가 3.78%로 목표(≤1%)를 위반한다.** 유형별 breakdown을 보면 원인이 뚜렷하다 — type0은 Validation 불량이 8건뿐이라 임계값이 Validation에는 맞았어도(허용 미스 0건) Test에서 Slip Rate 16.5%로 완전히 무너졌다. type2(Val 불량 32건), type3(24건)도 각각 1.7%, 7.1%로 목표를 넘겼다. 반대로 Validation 불량이 가장 많은 type1(239건)은 Test Slip Rate 1.02%로 목표에 가장 근접했다.
- 즉 **표본이 많은 유형일수록 유형별 임계값 선택이 안정적이고, 표본이 적을수록 Validation에 과적합돼 Test에서 깨진다** — `0823_lsw_002_baseline`과 `0824_lsw_003`에서 반복 관찰된 동일 패턴이다.

### 다음 단계

- 5분리 구조의 "자동화 효과는 크지만 안전기준 위반" 문제를 먼저 풀어야, 불균형 처리(Phase 2)로 넘어가는 게 의미가 있다. 후보:
  1. 임계값 선택을 유형별이 아니라 **전체 데이터 합산 Validation 기준**으로 하나만 고르고, 각 유형 모델에 동일하게 적용 (표본 수 문제를 pooled로 우회).
  2. 표본이 적은 유형(0, 2, 3)만 별도로 더 보수적인(안전 마진을 둔) 임계값 규칙 적용.
- 이번 실험은 구조만 비교하기 위해 `mapping.json` 마스킹을 적용하지 않았다 — 다음 feature selection 단계에서 마스킹을 넣었을 때 유형별 Test 안정성이 개선되는지도 함께 확인할 가치가 있다.
